In [ ]:
# Tải tập dữ liệu từ GitHub
!git clone "https://github.com/thaortrinh/pm25-hcmc.git"

# Cài đặt các thư viện cần thiết cho mô hình
!pip install catboost xgboost

fatal: destination path 'pm25-hcmc' already exists and is not an empty directory.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Đọc dữ liệu
df = pd.read_csv("./pm25-hcmc/data/processed/pm25_weather_merged.csv")
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values('datetime')

# Tạo biến mục tiêu: Dự báo PM2.5 cho 1 giờ tiếp theo
df['target_next_hour'] = df['pm25_avg'].shift(-1)

# Loại bỏ dòng cuối cùng (NaN do shift) và các cột metadata không cần thiết
df_baseline = df.dropna(subset=['target_next_hour']).copy()

# Chọn các cột đặc trưng "gốc" từ danh sách bạn cung cấp
# Chúng ta sẽ giữ lại các yếu tố khí tượng và thời gian cơ bản
features = [
    'pm25_avg', 'temperature_2m', 'relative_humidity_2m', 'precipitation',
    'wind_speed_10m', 'wind_direction_10m', 'surface_pressure',
    'boundary_layer_height', 'wind_u', 'wind_v',
    'weekday', 'year', 'month', 'day', 'hour', 'is_weekend'
]

X = df_baseline[features]
y = df_baseline['target_next_hour']

In [ ]:
# Chia dữ liệu theo tỷ lệ 70% Train - 15% Valid - 15% Test (theo đúng yêu cầu của bạn)
n = len(df_baseline)
train_end = int(n * 0.7)
valid_end = int(n * 0.85)

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_valid, y_valid = X.iloc[train_end:valid_end], y.iloc[train_end:valid_end]
X_test, y_test = X.iloc[valid_end:], y.iloc[valid_end:]

# Chuẩn hóa dữ liệu (Rất quan trọng cho Linear Regression và KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)

print(f"Kích thước tập Train: {X_train.shape}")
print(f"Kích thước tập Valid: {X_valid.shape}")
print(f"Kích thước tập Test: {X_test.shape}")

Kích thước tập Train: (7285, 16)
Kích thước tập Valid: (1561, 16)
Kích thước tập Test: (1562, 16)


In [ ]:
# Định nghĩa danh sách mô hình
models = {
    "Linear Regression": LinearRegression(),
    "K-Nearest Neighbors": KNeighborsRegressor(n_neighbors=5),
    "Random Forest": RandomForestRegressor(random_state=42),
    "XGBoost": XGBRegressor(random_state=42),
    "CatBoost": CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    l2_leaf_reg=5,
    random_strength=1,
    subsample=0.8,
    bootstrap_type="Bernoulli",
    od_type="Iter",
    od_wait=100,
    verbose=100
)
}

results = []

# Vòng lặp huấn luyện và đánh giá
for name, model in models.items():
    # Với LR và KNN dùng dữ liệu đã chuẩn hóa, các mô hình cây dùng dữ liệu gốc
    if name in ["Linear Regression", "K-Nearest Neighbors"]:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    # Tính toán 4 chỉ số đánh giá
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Model": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2 Score": r2
    })

# Hiển thị kết quả dưới dạng bảng
df_comparison = pd.DataFrame(results)
print("--- KẾT QUẢ CÁC MÔ HÌNH BASELINE (CHƯA CÓ FEATURE ENGINEERING) ---")
display(df_comparison.sort_values(by="R2 Score", ascending=False))

0:	learn: 15.8972064	total: 4.06ms	remaining: 8.12s
100:	learn: 7.6982891	total: 303ms	remaining: 5.69s
200:	learn: 7.2595372	total: 602ms	remaining: 5.39s
300:	learn: 7.0391708	total: 912ms	remaining: 5.15s
400:	learn: 6.8345325	total: 1.2s	remaining: 4.79s
500:	learn: 6.6560732	total: 1.49s	remaining: 4.46s
600:	learn: 6.5019664	total: 1.78s	remaining: 4.14s
700:	learn: 6.3610241	total: 2.1s	remaining: 3.88s
800:	learn: 6.2325992	total: 2.39s	remaining: 3.58s
900:	learn: 6.1093146	total: 2.69s	remaining: 3.28s
1000:	learn: 5.9997247	total: 3s	remaining: 3s
1100:	learn: 5.8848998	total: 3.3s	remaining: 2.7s
1200:	learn: 5.7767077	total: 3.59s	remaining: 2.39s
1300:	learn: 5.6801524	total: 3.88s	remaining: 2.08s
1400:	learn: 5.5840207	total: 4.2s	remaining: 1.79s
1500:	learn: 5.4945125	total: 4.5s	remaining: 1.49s
1600:	learn: 5.4038844	total: 4.79s	remaining: 1.19s
1700:	learn: 5.3204058	total: 5.11s	remaining: 898ms
1800:	learn: 5.2364392	total: 5.41s	remaining: 598ms
1900:	learn: 5.

,Model,MAE,MSE,RMSE,R2 Score
4,CatBoost,4.847723,67.225805,8.199134,0.746954
2,Random Forest,4.922505,69.422555,8.332020,0.738685
0,Linear Regression,4.989951,71.732775,8.469520,0.729989
3,XGBoost,5.391648,80.047394,8.946921,0.698692
1,K-Nearest Neighbors,7.043126,105.513719,10.271987,0.602833
